#Crédito Inclusivo (grupo Kurumin):

Este notebook junta, em ordem, todas as partes do grupo. Rode as células de cima pra baixo.

| Parte | O que faz | Arquivo em `partes/` |
|---|---|---|
| 1. Base do Banco Central (SCR) | Baixa e arruma os dados do SCR | `01_base_download_extracao`, `02_base_leitura_tratamento` |
| 2. Inadimplência alta vs recuperação rápida | Gráficos por UF e por região | `03_base_inadimplencia_alta_vs_recuperacao_rapida` |
| 3. Dívida saudável vs crítica | Gráficos por UF e por região | `04_base_divida_saudavel_vs_critica` |
| 4. Taxa de juros | Compara as instituições financeiras | `05_taxa_juros` |
| 5. Apostas (Bets) | Depósitos e apostadores por mês | `06_bets` |
| 6. IBGE | População e renda por estado | `07_ibge_populacao_renda` |

---
# Análise de Risco de Crédito

   1. Baixa os dados abertos do Banco Central (SCR), um .zip por ano.
   2. Abre os .zip e lê os .csv de dentro.
   3. Deixa só o que interessa (Pessoa Física) e arruma as colunas.
   4. Faz as duas análises que já estão prontas:
        - Inadimplência alta vs recuperação rápida
        - Dívida saudável vs crítica
      Cada uma desenha um gráfico no Colab (pra gente conferir os números).

---

Bibliotecas


In [ ]:
import pandas as pd                 # tabelas (DataFrames): é com ela que a gente soma, filtra e agrupa
import requests                     # baixar arquivos e falar com o servidor pela internet
import zipfile                      # abrir arquivos .zip
import os                           # criar pastas e listar arquivos
import time                         # esperar alguns segundos entre uma tentativa e outra de download
import matplotlib.pyplot as plt     # desenhar os gráficos
from concurrent.futures import ThreadPoolExecutor as tpe   # baixar vários arquivos ao mesmo tempo
from tqdm.auto import tqdm          # barrinha de progresso nos laços demorados

---

Configurações


In [ ]:
# Anos que vão ser baixados do Banco Central (quanto mais anos, mais demora)
anos = list(range(2024, 2026 + 1))   # range(2024, 2027) dá 2024, 2025 e 2026

# Pastas usadas no Colab
pasta_ext = "/content/dados_scr_extraidos"   # onde os .csv extraídos ficam

# Cria as pastas se elas ainda não existirem (exist_ok=True evita erro se já existirem)
os.makedirs(pasta_ext, exist_ok=True)

---

Download e extração dos arquivos do Banco Central


In [ ]:
def baixar_zip(ano):
    # Monta o endereço do .zip daquele ano
    url = f"https://www.bcb.gov.br/pda/desig/scrdata_{ano}.zip"

    # Caminho onde o .zip vai ficar salvo no Colab
    caminho = f"/content/scrdata_{ano}.zip"

    # Se a internet oscilar, tentamos de novo em vez de desistir de cara
    tentativas = 5

    for tentativa in range(1, tentativas + 1):
        try:
            # stream=True baixa aos poucos, sem carregar o arquivo inteiro na memória
            resposta = requests.get(url, stream=True)
            # Se o site respondeu com erro (404, 500...), isso gera uma exceção
            resposta.raise_for_status()

            # "wb" = escrever em modo binário (.zip não é texto)
            with open(caminho, "wb") as arquivo:
                # Grava o arquivo em pedaços de 1 MB (1024 * 1024 bytes)
                for pedaco in resposta.iter_content(chunk_size=1024 * 1024):
                    arquivo.write(pedaco)

            return caminho   # deu certo: devolve o caminho e sai da função

        except Exception as erro:
            # Deu erro: avisa qual tentativa falhou
            print(f"Ano {ano}: tentativa {tentativa} de {tentativas} falhou ({erro}).")
            if tentativa < tentativas:
                time.sleep(3)   # espera 3 segundos antes de tentar de novo

    # Chegou aqui = as 5 tentativas falharam. Devolve None (vazio) pra pular esse ano
    print(f"Não foi possível baixar {ano}. Pulando este ano.")
    return None

In [ ]:
# Baixa os anos ao mesmo tempo pra ir mais rápido.
# ThreadPoolExecutor é um "grupo de ajudantes": cada um baixa um ano.
# max_workers=3 -> no máximo 3 downloads rodando juntos, um por ano.
# executor.map(baixar_zip, anos) -> chama baixar_zip(ano) pra cada ano da lista
# e devolve os resultados na mesma ordem dos anos.
with tpe(max_workers=3) as executor:
    resultados = list(tqdm(executor.map(baixar_zip, anos), total=len(anos), desc="Baixando anos"))

# Fica só com os que deram certo (os anos que falharam voltam como None)
zips_baixados = [caminho for caminho in resultados if caminho is not None]
print("Anos baixados com sucesso:", zips_baixados)

# Abre cada .zip e joga os .csv de dentro na pasta de extração
for caminho_zip in tqdm(zips_baixados, desc="Extraindo arquivos ZIP"):
    with zipfile.ZipFile(caminho_zip, "r") as zip_aberto:   # "r" = só leitura
        zip_aberto.extractall(pasta_ext)

# Lista só os arquivos que terminam em .csv
arquivos_csv = [nome for nome in os.listdir(pasta_ext) if nome.endswith(".csv")]
print("Total de arquivos CSV extraídos (todos os anos):", len(arquivos_csv))

---

Leitura dos CSVs


In [ ]:
# Colunas do CSV que interessam (as outras nem são carregadas)
colunas = [
    "data_base",                 # mês de referência
    "uf",                        # estado
    "cliente",                   # PF (pessoa física) ou PJ (pessoa jurídica)
    "porte",                     # faixa de renda (usada pra achar o público vulnerável)
    "carteira_a_vencer",
    "carteira_inadimplencia",    # dívida com atraso grande (inadimplência alta)
    "a_vencer_ate_90_dias",      # dívida que vence em até 90 dias (recuperação rápida)
    "carteira_ativa",            # total da carteira de crédito
    "modalidade",
    "submodalidade",             # tipo de crédito (cartão, financiamento de imóvel...)
    "ativo_problematico",
]

# Faixas de renda (coluna "porte" do SCR) que contam como público vulnerável
# ("Sem rendimento" entra porque quem não tem renda também é público vulnerável.)
lista_portes_vulneraveis = ["Sem rendimento", "Até 1 salário mínimo", "Mais de 1 a 2 salários mínimos"]

tabelas_mensais = []   # lista que vai guardar os pedaços lidos

for nome_arquivo in tqdm(arquivos_csv, desc="Lendo arquivos CSV"):
    # Junta a pasta com o nome do arquivo pra ter o caminho completo
    caminho = os.path.join(pasta_ext, nome_arquivo)

    #   sep=";"           -> colunas separadas por ponto e vírgula
    #   decimal=","       -> números com vírgula (12,5 em vez de 12.5)
    #   usecols=colunas   -> só as colunas da lista acima
    #   encoding          -> lê acentos corretamente
    #   chunksize=200_000 -> devolve o arquivo em pedaços de 200 mil linhas
    leitor = pd.read_csv(
        caminho, sep=";", decimal=",", usecols=colunas,
        encoding="utf-8-sig", low_memory=False, chunksize=200_000
    )

    # Guarda cada pedaço na lista
    for pedaco in leitor:
        tabelas_mensais.append(pedaco)

# Junta todos os pedaços numa tabela só. ignore_index=True refaz a numeração das linhas
tabela_credito = pd.concat(tabelas_mensais, ignore_index=True)
print("Formato da tabela (linhas, colunas):", tabela_credito.shape)

---

Tratamento dos dados


In [ ]:
# --- 5.1 Só Pessoa Física ---------------------------------------------
# .copy() cria uma cópia independente
tabela_pf = tabela_credito[tabela_credito["cliente"] == "PF"].copy()
print("Linhas de Pessoa Física:", tabela_pf.shape[0])

In [ ]:
# --- 5.2 Coluna "publico" ---------------------------------------------
# "vulneravel" = renda até 2 salários mínimos; "demais" = o resto.
# isin() dá True/False pra cada linha e map() troca True/False pelos nomes.
e_vulneravel = tabela_pf["porte"].isin(lista_portes_vulneraveis)
tabela_pf["publico"] = e_vulneravel.map({True: "vulneravel", False: "demais"})

In [ ]:
# --- 5.3 Regiões ------------------------------------------------------
# Dicionário que diz a que região cada UF pertence
mapa_regiao = {
    "AC": "Norte", "AP": "Norte", "AM": "Norte", "PA": "Norte",
    "RO": "Norte", "RR": "Norte", "TO": "Norte",
    "AL": "Nordeste", "BA": "Nordeste", "CE": "Nordeste", "MA": "Nordeste",
    "PB": "Nordeste", "PE": "Nordeste", "PI": "Nordeste", "RN": "Nordeste", "SE": "Nordeste",
    "DF": "Centro-Oeste", "GO": "Centro-Oeste", "MT": "Centro-Oeste", "MS": "Centro-Oeste",
    "ES": "Sudeste", "MG": "Sudeste", "RJ": "Sudeste", "SP": "Sudeste",
    "PR": "Sul", "RS": "Sul", "SC": "Sul"
}

# UFs do Norte e do Nordeste: é a opção padrão (0) do menu
lista_norte_nordeste = [uf for uf, regiao in mapa_regiao.items() if regiao in ["Norte", "Nordeste"]]


# Opções do menu de região. A posição na lista é o número que a pessoa digita
# (posição 0 = "Norte + Nordeste", posição 1 = "Norte", e assim por diante).
menu_regioes = ["Norte + Nordeste", "Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]

In [ ]:
def escolher_regiao():
    # Mostra o menu, pergunta o número e devolve (nome da região, lista de UFs dela)

    # Mostra uma linha por opção: "0. Norte + Nordeste", "1. Norte"...
    print("Escolha a região:")
    for numero, nome in enumerate(menu_regioes):
        print(f"  {numero}. {nome}")

    # Repete a pergunta até a pessoa digitar um número que existe no menu
    while True:
        texto = input("Digite o número (Enter = 0): ").strip()
        if texto == "":
            texto = "0"                                   # Enter sem digitar nada = padrão
        if texto.isdigit() and int(texto) < len(menu_regioes):
            break                                         # número válido, sai do laço
        print(f"'{texto}' não é uma opção. Digite um número de 0 a {len(menu_regioes) - 1}.")

    nome = menu_regioes[int(texto)]                       # nome da opção escolhida

    if nome == "Norte + Nordeste":
        ufs = lista_norte_nordeste                        # a lista pronta lá de cima
    else:
        ufs = [uf for uf, regiao in mapa_regiao.items() if regiao == nome]   # UFs só dessa região
    return nome, ufs

---

Inadimplência alta vs recuperação rápida, por UF


In [ ]:
# Soma as duas colunas para cada UF
tabela_inadimplencia = tabela_pf.groupby("uf").agg(
    inadimplencia=("carteira_inadimplencia", "sum"),   # total inadimplente
    a_vencer_90d=("a_vencer_ate_90_dias", "sum")       # total que vence em até 90 dias
).reset_index()                                         # uf volta a ser uma coluna normal

# Converte pra bilhões só pra ficar fácil de ler no gráfico (1e9 = 1 bilhão)
tabela_inadimplencia["inadimplencia_bi"] = tabela_inadimplencia["inadimplencia"] / 1e9
tabela_inadimplencia["a_vencer_90d_bi"] = tabela_inadimplencia["a_vencer_90d"] / 1e9

# Pergunta a região pelo menu (Enter = Norte + Nordeste)
regiao_inadimplencia, ufs_inadimplencia = escolher_regiao()

# Fica só com as UFs escolhidas e ordena da maior inadimplência pra menor
grafico_inadimplencia = tabela_inadimplencia[tabela_inadimplencia["uf"].isin(ufs_inadimplencia)]
grafico_inadimplencia = grafico_inadimplencia.sort_values(by="inadimplencia_bi", ascending=False)

plt.style.use("ggplot")   # estilo visual do gráfico

# Gráfico de barras: uma barra vermelha (inadimplência) e uma azul (90 dias) por UF
eixo = grafico_inadimplencia.set_index("uf")[["inadimplencia_bi", "a_vencer_90d_bi"]].plot(
    kind="bar", figsize=(12, 6), color=["#ef4444", "#4f46e5"], width=0.6
)

# Escreve o valor em cima de cada barra
for barra in eixo.patches:
    altura = barra.get_height()   # a altura da barra é o próprio valor
    eixo.annotate(f"{altura:.1f}B", (barra.get_x() + barra.get_width() / 2, altura),
                  ha="center", va="bottom", xytext=(0, 3), textcoords="offset points",
                  fontweight="bold", fontsize=7)

plt.title("Inadimplência Alta vs Recuperação Rápida, por UF (R$ bilhões)\nBase SCR - Banco Central do Brasil",
          fontweight="bold", fontsize=12)
plt.xlabel(f"Região {regiao_inadimplencia}", fontweight="bold")
plt.ylabel("R$ bilhões", fontweight="bold")
plt.xticks(rotation=0)                                       # nomes das UFs na horizontal
plt.legend(["Inadimplência (alta)", "A vencer em 90 dias (recuperação rápida)"])
plt.ylim(0, grafico_inadimplencia[["inadimplencia_bi", "a_vencer_90d_bi"]].values.max() * 1.15)  # 15% de folga pro número não cortar
plt.savefig("/content/grafico_inadimplencia_uf.png", bbox_inches="tight")   # salva a imagem
plt.show()

---

Inadimplência alta vs recuperação rápida, por região


In [ ]:
# Põe em cada UF o nome da sua região (usando o dicionário mapa_regiao)
tabela_inadimplencia["regiao"] = tabela_inadimplencia["uf"].map(mapa_regiao)

# Soma as UFs de cada região
inadimplencia_regiao = tabela_inadimplencia.groupby("regiao").agg(
    inadimplencia_bi=("inadimplencia_bi", "sum"),
    a_vencer_90d_bi=("a_vencer_90d_bi", "sum")
).reset_index()

eixo = inadimplencia_regiao.set_index("regiao")[["inadimplencia_bi", "a_vencer_90d_bi"]].plot(
    kind="bar", figsize=(8, 5), color=["#ef4444", "#4f46e5"], width=0.6
)
for barra in eixo.patches:
    altura = barra.get_height()
    eixo.annotate(f"{altura:.1f}B", (barra.get_x() + barra.get_width() / 2, altura),
                  ha="center", va="bottom", xytext=(0, 3), textcoords="offset points",
                  fontweight="bold", fontsize=8)

plt.title("Inadimplência Alta vs Recuperação Rápida, por Região (R$ bilhões)", fontweight="bold", fontsize=12)
plt.xlabel("Região", fontweight="bold")
plt.ylabel("R$ bilhões", fontweight="bold")
plt.xticks(rotation=0)
plt.legend(["Inadimplência (alta)", "A vencer em 90 dias (recuperação rápida)"])
plt.ylim(0, inadimplencia_regiao[["inadimplencia_bi", "a_vencer_90d_bi"]].values.max() * 1.15)
plt.savefig("/content/grafico_inadimplencia_regiao.png", bbox_inches="tight")
plt.show()


---

% de dívida saudável vs crítica, por UF


In [ ]:
# Submodalidades consideradas saudáveis
submod_saudaveis = [
    "Aquisição de bens - veículos automotores",
    "Aquisição de bens com interveniência - veículos autom.",
    "Arrendamento financeiro de veículos automotores",
    "Arrendamento financeiro exceto veículos automotores e imóveis",
    "Financiamento de projeto",
    "Financiamento habitacional - SFH",
    "Financiamento habitacional - exceto SFH",
    "Financiamento imobiliário - empreendim, exceto habitac.",
    "Financiamentos agroindustriais",
    "Home Equity"
]

# Submodalidades consideradas críticas
submod_criticas = [
    "Antecipação de fatura de cartão de crédito",
    "Aquisição de bens com interveniência - outros bens",
    "Cartão de crédito - compra à vista e parcelado lojista",
    "Cartão de crédito - compra, fatura parcelada ou saque financiado pela instituição financeira emitente do cartão",
    "Cheque especial",
    "Crédito pessoal - com consignação em folha de pagam.",
    "Crédito pessoal - sem consignação em folha de pagam.",
    "Crédito rotativo vinculado a cartão de crédito",
    "Microcrédito",
    "Outros com característica de crédito",
    "Outros empréstimos",
    "Outros financiamentos"
]

# Alguns nomes vêm com um travessão ("\x96") no lugar do hífen.
# Aqui tiramos espaços das pontas e trocamos o travessão por hífen, senão
# o nome não bate com as listas acima.
submod_limpa = tabela_pf["submodalidade"].str.strip().str.replace("\x96", "-", regex=False)

# Cria duas colunas novas: o valor da carteira só nas linhas saudáveis (ou críticas).
# .where(condição, 0) mantém o valor quando a condição é verdadeira e põe 0 quando não é.
tabela_pf["carteira_saudavel"] = tabela_pf["carteira_ativa"].where(submod_limpa.isin(submod_saudaveis), 0)
tabela_pf["carteira_critica"] = tabela_pf["carteira_ativa"].where(submod_limpa.isin(submod_criticas), 0)

# Soma por UF
tabela_divida = tabela_pf.groupby("uf").agg(
    carteira_saudavel=("carteira_saudavel", "sum"),
    carteira_critica=("carteira_critica", "sum"),
    carteira_ativa_total=("carteira_ativa", "sum")
).reset_index()

# Porcentagem da carteira total que é saudável e que é crítica
tabela_divida["percentual_saudavel"] = (tabela_divida["carteira_saudavel"] / tabela_divida["carteira_ativa_total"]) * 100
tabela_divida["percentual_critica"] = (tabela_divida["carteira_critica"] / tabela_divida["carteira_ativa_total"]) * 100

# O que não está em nenhuma das duas listas
tabela_divida["percentual_outra"] = 100 - tabela_divida["percentual_saudavel"] - tabela_divida["percentual_critica"]

# Pergunta a região e filtra as UFs, da menos crítica pra mais crítica
regiao_divida, ufs_divida = escolher_regiao()
grafico_divida = tabela_divida[tabela_divida["uf"].isin(ufs_divida)].sort_values(by="percentual_critica")

# Barras uma ao lado da outra: verde = saudável, vermelho = crítica
eixo = grafico_divida.set_index("uf")[["percentual_saudavel", "percentual_critica"]].plot(
    kind="bar", figsize=(12, 6), color=["#10b981", "#ef4444"], width=0.6, legend=False
)

for barra in eixo.patches:
    altura = barra.get_height()
    if altura > 1:   # só escreve se a barra for alta o bastante pra caber o número
        eixo.annotate(f"{altura:.0f}%", (barra.get_x() + barra.get_width() / 2, barra.get_y() + altura / 2),
                      ha="center", va="center", fontweight="bold", fontsize=8, color="white")

plt.title("% de Dívida Saudável vs Crítica, por UF\nBase SCR - Banco Central do Brasil", fontweight="bold", fontsize=12)
plt.xlabel(f"Região {regiao_divida}", fontweight="bold")
plt.ylabel("% da carteira ativa", fontweight="bold")
plt.xticks(rotation=0)
plt.legend(["Saudável", "Crítica"])
plt.ylim(0, 100)
plt.savefig("/content/grafico_divida_uf.png", bbox_inches="tight")
plt.show()

---

% de dívida saudável vs crítica, por região


In [ ]:
# Põe a região em cada UF e soma as UFs de cada região
tabela_divida["regiao"] = tabela_divida["uf"].map(mapa_regiao)
divida_regiao = tabela_divida.groupby("regiao").agg(
    carteira_saudavel=("carteira_saudavel", "sum"),
    carteira_critica=("carteira_critica", "sum"),
    carteira_ativa_total=("carteira_ativa_total", "sum")
).reset_index()

# Recalcula as porcentagens com os totais da região (não dá pra tirar média das % das UFs)
divida_regiao["percentual_saudavel"] = (divida_regiao["carteira_saudavel"] / divida_regiao["carteira_ativa_total"]) * 100
divida_regiao["percentual_critica"] = (divida_regiao["carteira_critica"] / divida_regiao["carteira_ativa_total"]) * 100
divida_regiao = divida_regiao.sort_values(by="percentual_critica")

eixo = divida_regiao.set_index("regiao")[["percentual_saudavel", "percentual_critica"]].plot(
    kind="bar", figsize=(8, 5), color=["#10b981", "#ef4444"], width=0.6, legend=False
)
for barra in eixo.patches:
    altura = barra.get_height()
    if altura > 1:
        eixo.annotate(f"{altura:.0f}%", (barra.get_x() + barra.get_width() / 2, barra.get_y() + altura / 2),
                      ha="center", va="center", fontweight="bold", fontsize=9, color="white")

plt.title("% de Dívida Saudável vs Crítica, por Região", fontweight="bold", fontsize=12)
plt.xlabel("Região", fontweight="bold")
plt.ylabel("% da carteira ativa", fontweight="bold")
plt.xticks(rotation=0)
plt.legend(["Saudável", "Crítica"])
plt.ylim(0, 100)
plt.savefig("/content/grafico_divida_regiao.png", bbox_inches="tight")
plt.show()

---
# Taxa de juros das maiores instituições financeiras

**O que este notebook faz**
1. Baixa do Banco Central (API Olinda) as taxas de juros mensais.
2. Deixa só o financiamento imobiliário "com taxas de mercado - Pós-fixado referenciado em TR" e só as 8 instituições que o card pede.
3. Mostra dois gráficos:
   - barras: comparação das taxas no mês mais recente
   - linhas: evolução mês a mês de cada instituição (desde jan/2024)

**Por que financiamento imobiliário pós-fixado em TR?**

A API traz vários tipos de financiamento imobiliário (com taxas de mercado ou reguladas; prefixado, pós-fixado em TR, pós-fixado em IPCA...). Pra comparar os bancos de forma justa, a gente olha o MESMO produto em todos.

TR é a Taxa Referencial, um índice calculado pelo Banco Central. Nesse tipo de financiamento a dívida é corrigida pela TR e o banco cobra os juros por cima. Se misturássemos produtos diferentes, um banco poderia parecer "mais caro" só porque a gente pegou um produto mais caro dele.

---
Configurações da taxa de juros

In [ ]:
# Endereço da API de taxas de juros do Banco Central (o mesmo que está no card)
url_juros = "https://olinda.bcb.gov.br/olinda/servico/taxaJuros/versao/v2/odata/TaxasJurosMensalPorMes"

# Colunas que a gente pede pra API
colunas_api = "Mes,Modalidade,InstituicaoFinanceira,TaxaJurosAoMes,anoMes"

# Pedaço do nome da modalidade que a gente quer.
# O bloco 3 mostra quais modalidades foram encontradas, pra gente conferir se pegou só a certa.
parte_modalidade = "Pós-fixado referenciado em TR"

# A API tem dois "tipos" de financiamento imobiliário: "com taxas de mercado" e
# "com taxas reguladas" (as reguladas seguem regras definidas pelo governo).
# Os dois têm uma versão pós-fixada em TR, então precisamos separar.
# O card pede as taxas de mercado.
parte_tipo = "taxas de mercado"

# As 8 instituições que devem aparecer nos gráficos (a lista vem do card)
instituicoes = [
    "BANCO SICOOB S.A.",
    "BANCO COOPERATIVO SICREDI",
    "ITAÚ UNIBANCO S.A.",
    "CAIXA ECONOMICA FEDERAL",
    "BCO SANTANDER (BRASIL) S.A.",
    "BCO BRADESCO S.A.",
    "BANCO INTER",
    "BCO DO BRASIL S.A.",
]

# Primeiro mês que aparece nos gráficos
mes_inicial = "2024-01"

---
Download das taxas de juros

In [ ]:
# A API não entrega tudo de uma vez: ela entrega em "páginas", como um livro.
# A gente faz o pedido mandando alguns parâmetros no final do endereço.
# Eles começam com "$" porque é o padrão dessa API (chama OData): o "$" só
# avisa "isto aqui é uma instrução pra API, não é um dado". Sem o "$" a API
# ignora o parâmetro. Os que usamos:
#   $top     = quantas linhas eu quero receber neste pedido   (tamanho da página)
#   $skip    = quantas linhas eu quero PULAR antes de começar (onde a página começa)
#   $format  = em que formato quero a resposta (json)
#   $select  = quais colunas eu quero
#
# Exemplo com $top=5000:
#   1º pedido: $skip=0     -> linhas 1 a 5000
#   2º pedido: $skip=5000  -> linhas 5001 a 10000
#   3º pedido: $skip=10000 -> linhas 10001 a 15000   ... e assim por diante
# Por isso precisa dos DOIS: o $top diz o tamanho da página e o $skip diz qual página.
# Quando a API devolve uma página vazia, os dados acabaram.

tamanho_pagina = 5000     # quantas linhas pedimos por vez
registros = []            # lista vazia onde vamos juntando todas as linhas baixadas
inicio = 0                # quantas linhas já baixamos (é o valor do "$skip")

# O "for" repete o pedido, uma página por volta.
# range(400) é uma trava de segurança: no máximo 400 páginas.
# Assim, se a API se comportar de um jeito estranho, o laço não roda pra sempre.
for numero_pagina in range(400):
    resposta = requests.get(url_juros, params={      # faz o pedido (GET) pra API
        "$top": tamanho_pagina,                      # quantas linhas queremos
        "$skip": inicio,                             # quantas linhas pular
        "$format": "json",                           # queremos a resposta em JSON
        "$select": colunas_api,                      # só as colunas que a gente precisa
    })

    # Erro 403 = o site do Banco Central recusou o pedido. Isso já aconteceu quando o Colab nos deu uma máquina (IP) que o site não aceita.
    # Como resolver:
    # "Desconectar e excluir ambiente de execução"
    # reconectar e rodar de novo (a máquina nova costuma ter outro IP).
    if resposta.status_code == 403:
        raise RuntimeError("O Banco Central recusou o pedido (erro 403). Desconecte e exclua o ambiente de execução do Colab, reconecte e rode de novo.")   # para o código aqui e mostra essa mensagem

    resposta.raise_for_status()                      # para com erro se a API respondeu com outro problema (404, 500...)

    pagina = resposta.json().get("value", [])        # as linhas ficam dentro da chave "value" do JSON

    if len(pagina) == 0:                             # página vazia = acabaram os dados
        break                                        # sai do laço

    registros += pagina                              # junta as linhas dessa página na lista geral
    inicio += len(pagina)                            # a próxima página começa depois das linhas que já temos
    print(f"Linhas baixadas até agora: {len(registros)}")   # mostra o progresso

# Exemplo do que aparece (os números mudam quando o Banco Central atualiza): 5000 e depois 5953.
# A segunda página veio com menos de 5000 linhas porque os dados acabaram ali.
# Ou seja, o 5000 é só o tamanho da página, o total é o último número.

# Transforma a lista numa tabela do pandas (cada item da lista vira uma linha)
tabela_juros_bruta = pd.DataFrame(registros)
print("Formato da tabela baixada (linhas, colunas):", tabela_juros_bruta.shape)   # shape = (quantidade de linhas, quantidade de colunas)
print("Período baixado:", tabela_juros_bruta["anoMes"].min(), "até", tabela_juros_bruta["anoMes"].max())   # mês mais antigo e mais novo

---
Tratamento dos dados

In [ ]:
tabela_juros = tabela_juros_bruta.copy()   # trabalha numa cópia, pra não mexer na tabela original que foi baixada

# --- 3.1 Só a modalidade TR -----------------------------------------
# str.contains() procura um pedaço de texto dentro do nome da modalidade e devolve
# True ou False pra cada linha. case=False ignora maiúscula/minúscula e
# na=False trata campo vazio como "não achou".
e_tr = tabela_juros["Modalidade"].str.contains(parte_modalidade, case=False, na=False)      # tem "Pós-fixado referenciado em TR"?
e_imobiliario = tabela_juros["Modalidade"].str.contains("imobili", case=False, na=False)    # é financiamento imobiliário?
e_mercado = tabela_juros["Modalidade"].str.contains(parte_tipo, case=False, na=False)       # é "taxas de mercado"?
tabela_juros = tabela_juros[e_tr & e_imobiliario & e_mercado].copy()   # fica só com as linhas onde as TRÊS respostas são True

# Mostra quais modalidades sobraram. Tem que aparecer UMA só.
print("Modalidades encontradas:")
print(tabela_juros["Modalidade"].value_counts())    # conta quantas linhas tem cada modalidade

# --- 3.2 Só as instituições do card ------------------------------------
# O Banco Central escreve os nomes com pontuação ("S.A.", "(BRASIL)"). Pra comparar
# sem erro por causa de ponto ou parêntese, a gente "limpa" o nome dos DOIS lados
# (o que veio da API e a nossa lista) do mesmo jeito: maiúsculo, sem pontuação,
# sem espaço duplicado. Exemplo: "BANCO SICOOB S.A." vira "BANCO SICOOB SA".
def limpar_nome(nomes):                                         # "def" cria uma função: uma receita que a gente usa quantas vezes quiser
    return (nomes.str.upper()                                   # tudo maiúsculo
                 .str.replace(r"[^\w\s]", "", regex=True)       # tira tudo que não é letra, número ou espaço (ponto, parêntese, hífen...)
                 .str.replace(r"\s+", " ", regex=True)          # vários espaços seguidos viram um só
                 .str.strip())                                  # tira espaço do começo e do fim

nomes_limpos = limpar_nome(pd.Series(instituicoes))             # limpa a nossa lista de 8 (pd.Series é uma lista do pandas)
nome_bonito = dict(zip(nomes_limpos, instituicoes))             # dicionário "nome limpo" -> "nome original", pra mostrar o nome certo nos gráficos

tabela_juros["instituicao_limpa"] = limpar_nome(tabela_juros["InstituicaoFinanceira"])           # coluna nova com o nome limpo que veio da API
tabela_juros = tabela_juros[tabela_juros["instituicao_limpa"].isin(nomes_limpos)].copy()         # fica só com as linhas cujo nome está na nossa lista
tabela_juros["instituicao"] = tabela_juros["instituicao_limpa"].map(nome_bonito)                 # volta pro nome original (com "S.A.") que vai aparecer nos gráficos

# --- 3.3 Taxa como número ----------------------------------------------
# errors="coerce" troca o que não for número por vazio (NaN), e depois a gente descarta.
tabela_juros["taxa"] = pd.to_numeric(tabela_juros["TaxaJurosAoMes"], errors="coerce")   # garante que a taxa é número
tabela_juros = tabela_juros.dropna(subset=["taxa"])                                      # joga fora as linhas sem taxa

# --- 3.4 Só os meses a partir de mes_inicial ---------------------------
# A coluna anoMes vem como texto "2026-09" (ano-mês). Nesse formato a ordem
# alfabética é a mesma ordem do tempo, então podemos comparar com ">=".
tabela_juros = tabela_juros[tabela_juros["anoMes"].astype(str) >= mes_inicial]   # fica só de "2024-01" pra frente

# --- 3.5 Uma linha por instituição e mês -------------------------------
# Se a mesma instituição aparecer mais de uma vez no mesmo mês, avisa e tira a média.
repetidas = tabela_juros.duplicated(["anoMes", "instituicao"]).sum()   # conta as linhas repetidas (mesma instituição no mesmo mês)
if repetidas > 0:                                                      # se tiver alguma repetida...
    print(f"ATENÇÃO: {repetidas} linhas repetidas (mesma instituição no mesmo mês). Usando a média delas.")   # avisa quantas eram

# "Mes" é o nome do mês escrito pela API (ex.: "Set-2026"): é o que mostramos nos gráficos.
# "anoMes" (ex.: "2026-09") é só pra ORDENAR os meses.
# groupby junta as linhas que têm o mesmo mês e a mesma instituição e mean() tira a média da taxa.
# as_index=False mantém essas colunas como colunas normais da tabela.
tabela_juros = tabela_juros.groupby(["anoMes", "Mes", "instituicao"], as_index=False)["taxa"].mean()

if tabela_juros.empty:    # se não sobrou nenhuma linha, tem algo errado nos filtros
    raise RuntimeError("Nenhuma linha sobrou depois dos filtros. Confira a modalidade e os nomes das instituições no bloco 1.")   # para e explica

# --- 3.6 Conferência rápida --------------------------------------------
for nome in instituicoes:                                       # passa pelas 8 instituições
    if nome not in tabela_juros["instituicao"].values:          # alguma sumiu depois dos filtros?
        print(f"ATENÇÃO: não encontrei '{nome}' nos dados. Confira o nome escrito na lista.")   # avisa qual instituição sumiu

meses = sorted(tabela_juros["anoMes"].unique())                 # lista dos meses que sobraram, em ordem
print("Linhas tratadas:", tabela_juros.shape[0])   # quantas linhas ficaram na tabela final
print(f"Meses: {len(meses)} (de {meses[0]} até {meses[-1]})")   # quantos meses e qual o primeiro e o último
print("Linhas se TODAS as instituições tivessem taxa em TODOS os meses:", len(meses) * len(instituicoes))   # meses x instituições
print("Se tiver menos linhas que isso, a API não trouxe taxa de alguma instituição em algum mês (vira buraco no gráfico de linhas).")   # explicação pra quem está lendo o resultado
display(tabela_juros.head(10))                                  # mostra as 10 primeiras linhas da tabela tratada

---
Gráfico de barras: taxas no mês mais recente



In [ ]:
ultimo_mes = tabela_juros["anoMes"].max()   # mês mais recente, no formato "2026-09" (como é texto AAAA-MM, o maior é o mais novo)
nome_ultimo_mes = tabela_juros.loc[tabela_juros["anoMes"] == ultimo_mes, "Mes"].iloc[0]   # nome desse mês escrito pela API (ex.: "Set-2026"), pro título

# Só as linhas desse mês, da taxa mais alta pra mais baixa
# (sort_values ordena pela coluna "taxa" e ascending=False deixa do maior pro menor)
grafico_barras = tabela_juros[tabela_juros["anoMes"] == ultimo_mes].sort_values(by="taxa", ascending=False)

plt.style.use("ggplot")   # mesmo estilo visual dos outros gráficos (fundo cinza com grade)

# Desenha as barras: o nome da instituição vai embaixo e a taxa define a altura.
# kind="bar" é o gráfico de barras | figsize=(12, 6) é o tamanho (largura, altura)
# color é a cor das barras | width=0.6 é a largura de cada barra
eixo = grafico_barras.set_index("instituicao")["taxa"].plot(
    kind="bar", figsize=(12, 6), color="#4f46e5", width=0.6
)

# Escreve o valor em cima de cada barra.
# eixo.patches é a lista das barras do gráfico.
for barra in eixo.patches:                                          # passa por cada barra
    altura = barra.get_height()                                     # altura da barra = valor da taxa
    eixo.annotate(f"{altura:.2f}%",                                 # texto: o valor com 2 casas decimais e o símbolo %
                  (barra.get_x() + barra.get_width() / 2, altura),  # onde escrever: no meio da barra, na altura do topo
                  ha="center", va="bottom",                         # centraliza na horizontal e deixa o texto logo acima do ponto
                  xytext=(0, 3), textcoords="offset points",        # afasta o texto 3 pontos pra cima, pra não colar na barra
                  fontweight="bold", fontsize=9)                    # negrito, tamanho 9

plt.title(f"Taxa de Juros ao Mês por Instituição - Financiamento Imobiliário (taxas de mercado, pós-fixado TR) | {nome_ultimo_mes}\nBase Olinda - Banco Central do Brasil",   # título em 2 linhas (\n pula linha); já mostra o mês de referência
          fontweight="bold", fontsize=11)                           # negrito, tamanho 11
plt.xlabel("Instituição financeira", fontweight="bold")             # legenda do eixo horizontal
plt.ylabel("Taxa de juros ao mês (%)", fontweight="bold")           # legenda do eixo vertical
plt.xticks(rotation=45, ha="right")                                 # nomes inclinados em 45 graus pra caberem
plt.ylim(0, grafico_barras["taxa"].max() * 1.15)                    # eixo vertical de 0 até o maior valor + 15%, pra o número não cortar
plt.tight_layout()                                                  # ajusta as margens pra nada ficar cortado
# plt.savefig("/content/grafico_taxa_juros_barras.png", bbox_inches="tight")   # salva a imagem
plt.show()                                                          # mostra o gráfico na tela

---
Gráfico de linhas: evolução mês a mês

In [ ]:
# Vira uma tabela com UMA LINHA POR MÊS e UMA COLUNA POR INSTITUIÇÃO
# (é o formato que o .plot() precisa pra desenhar uma linha por instituição).
# index = o que vai nas linhas | columns = o que vira coluna | values = o que preenche a tabela
tabela_linhas = tabela_juros.pivot(index="anoMes", columns="instituicao", values="taxa")

# Troca "2026-09" pelo nome do mês da API ("Set-2026") só pra aparecer no gráfico.
# Como a tabela já está em ordem de anoMes, a ordem dos meses continua certa.
nomes_meses = tabela_juros.drop_duplicates("anoMes").set_index("anoMes")["Mes"]   # tabela de conversão "2026-09" -> "Set-2026"
tabela_linhas.index = tabela_linhas.index.map(nomes_meses)                         # aplica a conversão nas linhas

# Quantos meses cada instituição ficou SEM taxa nessa modalidade (vazio = NaN).
# Vazio quer dizer que a API não trouxe taxa dessa instituição naquele mês.
print("Meses sem taxa por instituição:")
print(tabela_linhas.isna().sum())   # isna() marca os vazios e sum() conta quantos tem em cada coluna

plt.style.use("ggplot")   # mesmo estilo dos outros gráficos (repetido aqui pra esta célula funcionar sozinha)

# Desenha uma linha por instituição.
# marker="o" põe uma bolinha em cada mês.
# colormap="tab10" usa 10 cores bem diferentes. O padrão do ggplot só tem 7 cores,
# e com 8 instituições duas ficariam com a mesma cor.
# Onde falta taxa a linha fica quebrada. A gente NÃO preenche o buraco pra não inventar dado.
eixo = tabela_linhas.plot(kind="line", marker="o", figsize=(12, 6), colormap="tab10")

# Guarda a cor de cada linha, pra escrever o valor final com a mesma cor da linha
cor_da_linha = {linha.get_label(): linha.get_color() for linha in eixo.get_lines()}

# Adiciona a porcentagem em cima de cada ponto do gráfico
#for col in tabela_linhas.columns:
#    cor = cor_da_linha[col]
#    for i, valor in enumerate(tabela_linhas[col]):
#        if pd.notna(valor):
#            # i é a posição horizontal (0, 1, 2...)
#            eixo.text(i, valor + 0.005, f"{valor:.2f}%",
#                      color=cor, fontsize=7, ha="center", va="bottom", alpha=0.85)

# Valor do ÚLTIMO mês de cada instituição, do maior pro menor (dropna ignora as que estão vazias)
ultimos_valores = tabela_linhas.iloc[-1].dropna().sort_values(ascending=False)

# Os valores finais são muito parecidos (ex.: 0.97, 0.97, 0.96, 0.95). Se a gente escrevesse
# cada número exatamente na altura da linha, eles ficariam um em cima do outro.
# Então, indo do maior pro menor, a gente garante uma distância mínima entre os textos.
distancia_minima = 0.016    # distância mínima entre dois textos, na escala do eixo vertical
altura_anterior = None      # altura do texto anterior (começa sem nenhum)
posicao_do_texto = {}       # aqui guardamos a altura onde cada texto vai ficar
for instituicao, valor in ultimos_valores.items():                      # passa pelas instituições, do maior valor pro menor
    altura = valor                                                      # começa na altura real da linha
    if altura_anterior is not None and altura_anterior - altura < distancia_minima:   # tem texto em cima e ficou perto demais?
        altura = altura_anterior - distancia_minima                     # ficou perto demais do texto de cima: empurra um pouco pra baixo
    posicao_do_texto[instituicao] = altura                              # guarda a altura escolhida
    altura_anterior = altura                                            # esse texto vira o "anterior" do próximo

# Escreve o valor final de cada instituição, à direita do último ponto
ultima_posicao = len(tabela_linhas) - 1                                 # posição do último mês no eixo horizontal (0, 1, 2...)
for instituicao, valor in ultimos_valores.items():                      # de novo, uma instituição por vez
    eixo.text(ultima_posicao + 0.4, posicao_do_texto[instituicao],      # x: logo depois do último ponto | y: altura que calculamos
              f"{valor:.2f}%",                                          # texto: valor com 2 casas decimais e %
              color=cor_da_linha[instituicao],                          # mesma cor da linha da instituição
              va="center", fontsize=8, fontweight="bold")               # centralizado na vertical, tamanho 8, negrito
eixo.set_xlim(-0.5, ultima_posicao + 2.5)                               # deixa espaço à direita pros textos não cortarem

plt.title("Evolução da Taxa de Juros ao Mês - Financiamento Imobiliário (taxas de mercado, pós-fixado TR)\nBase Olinda - Banco Central do Brasil",   # título em 2 linhas
          fontweight="bold", fontsize=11)                               # negrito, tamanho 11
plt.xlabel("Mês", fontweight="bold")                                    # legenda do eixo horizontal
plt.ylabel("Taxa de juros ao mês (%)", fontweight="bold")               # legenda do eixo vertical
plt.xticks(range(len(tabela_linhas)), tabela_linhas.index, rotation=90, fontsize=8)   # mostra TODOS os meses, de pé, em letra pequena
plt.legend(title="Instituição", bbox_to_anchor=(1.02, 1), loc="upper left")           # legenda fora do gráfico, no canto de cima à direita
plt.tight_layout()                                                      # ajusta as margens pra nada ficar cortado
plt.savefig("/content/grafico_taxa_juros_linhas.png", bbox_inches="tight")   # salva a imagem
plt.show()                                                              # mostra o gráfico na tela

# A tabela com todos os valores do gráfico (pra conferir os números)
display(tabela_linhas.round(2))

---
# Apostas (Bets): depósitos e apostadores por mês

**O que este notebook faz**
1. Baixa do endpoint do grupo (mockapi) os dados mensais do Ministério da Fazenda: quanto foi depositado nas bets e quantos apostadores teve.
2. Arruma os dados (nome das colunas, meses, unidades).
3. Mostra a evolução mês a mês em dois gráficos.

**De onde vêm os dados**

Não existem bases abertas sobre apostas. Por isso o grupo copiou os dados do Ministério da Fazenda pra um endpoint (mockapi), que devolve um JSON. Aqui a gente faz um GET nesse endereço.

---
Configurações das apostas

In [ ]:
# Endereço do endpoint (mockapi) onde o grupo guardou os dados do Ministério da Fazenda.
# Ele devolve um JSON com um item por mês. Cada item tem: id, Mes (ex.: "01/2025"),
# Numero-apostadores e Deposito-mes (em reais).
url_bets = "https://6aa1d1d4ccb3db9689a5cff3.mockapi.io/endividamento-mf-2026"

---
Download dos dados

In [ ]:
resposta_bets = requests.get(url_bets, timeout=30)   # faz o pedido (GET) pro endpoint; timeout=30 = desiste se ele não responder em 30 segundos
resposta_bets.raise_for_status()                      # para com erro se o endpoint respondeu com problema (404, 500...)

# Esse endpoint já devolve uma LISTA direto, um item por mês.
# (Diferente da API do Banco Central, que coloca as linhas dentro da chave "value".)
registros_bets = resposta_bets.json()

tabela_bets_bruta = pd.DataFrame(registros_bets)      # vira uma tabela do pandas: cada item da lista vira uma linha
print("Formato da tabela baixada (linhas, colunas):", tabela_bets_bruta.shape)   # shape = (quantidade de linhas, quantidade de colunas)
display(tabela_bets_bruta)                            # mostra a tabela do jeito que veio do endpoint

---
Tratamento dos dados

In [ ]:
tabela_bets = tabela_bets_bruta.copy()   # trabalha numa cópia, pra não mexer na tabela original que foi baixada

# --- 3.1 Nomes das colunas ---------------------------------------------
# Os nomes originais têm hífen ("Numero-apostadores").
# rename troca o nome de antes pelo nome de depois.
tabela_bets = tabela_bets.rename(columns={"Numero-apostadores": "apostadores", "Deposito-mes": "deposito"})

# --- 3.2 Tipos: trocando string por número ---------------------
tabela_bets["id"] = tabela_bets["id"].astype(int)                                          # o id veio como texto ("1", "2"...), vira número inteiro
tabela_bets["apostadores"] = pd.to_numeric(tabela_bets["apostadores"], errors="coerce")    # errors="coerce": o que não for número vira vazio (NaN)
tabela_bets["deposito"] = pd.to_numeric(tabela_bets["deposito"], errors="coerce")          # mesma coisa pro valor depositado
tabela_bets = tabela_bets.dropna(subset=["apostadores", "deposito"])                       # joga fora as linhas que ficaram sem número
tabela_bets = tabela_bets.sort_values("id").reset_index(drop=True)                         # põe na ordem do id e refaz a numeração das linhas (0, 1, 2...)

# --- 3.3 Meses ----------------------------------------------------------
# O mês vem como texto "01/2025". Vira uma data de verdade (%m = mês, %Y = ano com 4 dígitos).
# Com data de verdade o pandas sabe ordenar e comparar.
tabela_bets["data"] = pd.to_datetime(tabela_bets["Mes"], format="%m/%Y")
tabela_bets = tabela_bets.sort_values("data").reset_index(drop=True)    # garante a ordem do tempo, do mês mais antigo pro mais novo

# Confere se, depois da correção, ainda sobrou mês repetido ou faltando
if tabela_bets["data"].duplicated().any():
    print("ATENÇÃO: ainda existe mês repetido na tabela.")   # só aparece se a correção não resolveu
todos_os_meses = pd.date_range(tabela_bets["data"].min(), tabela_bets["data"].max(), freq="MS")   # todos os meses entre o primeiro e o último
faltando = todos_os_meses.difference(pd.DatetimeIndex(tabela_bets["data"]))                        # os que não estão na tabela
if len(faltando) > 0:                                                                                  # se faltar algum mês...
    print("ATENÇÃO: faltam estes meses:", [mes.strftime("%m/%Y") for mes in faltando])   # mostra quais

# --- 3.4 Colunas prontas pros gráficos ---------------------------------
# Os valores são enormes (12345678901.23 reais). Nos gráficos fica melhor em bilhões e em milhões.
tabela_bets["mes"] = tabela_bets["data"].dt.strftime("%m/%Y")                  # texto do mês que vai no eixo do gráfico (ex.: "05/2025")
tabela_bets["deposito_bi"] = tabela_bets["deposito"] / 1_000_000_000           # reais -> bilhões de reais
tabela_bets["apostadores_mi"] = tabela_bets["apostadores"] / 1_000_000         # pessoas -> milhões de pessoas

# --- 3.5 Conferência rápida --------------------------------------------
print("Meses na tabela:", len(tabela_bets))                                                  # quantos meses ficaram
print("Período:", tabela_bets["mes"].iloc[0], "até", tabela_bets["mes"].iloc[-1])           # primeiro e último mês
display(tabela_bets[["mes", "deposito_bi", "apostadores_mi"]].round(2))                      # tabela final (2 casas decimais)

---
Gráficos: evolução mês a mês

In [ ]:
plt.style.use("ggplot")   # mesmo estilo visual dos outros gráficos (fundo cinza com grade)

# Tabela só com o que vai pro gráfico: o índice (nome do mês) vai no eixo horizontal
# e cada coluna vira um gráfico.
tabela_grafico = tabela_bets.set_index("mes")[["deposito_bi", "apostadores_mi"]]

# Dois gráficos um embaixo do outro. Em vez de um gráfico só com dois eixos verticais
# (que confunde, porque as escalas são diferentes), cada série tem o seu.
# kind="line" = linhas | marker="o" = bolinha em cada mês
# subplots=True = um gráfico por coluna | sharex=True = os dois dividem o eixo dos meses
# legend=False = sem legenda (cada gráfico já tem título) | color = uma cor pra cada gráfico
eixos = tabela_grafico.plot(kind="line", marker="o", subplots=True, sharex=True,
                            figsize=(12, 8), legend=False, color=["#4f46e5", "#dc2626"])

# Escreve o valor em cima de cada ponto.
# enumerate dá a posição do ponto (0, 1, 2...) junto com o valor dele.
for posicao, valor in enumerate(tabela_grafico["deposito_bi"]):                     # passa por cada mês do primeiro gráfico
    eixos[0].annotate(f"{valor:.1f}", (posicao, valor), textcoords="offset points",   # texto com 1 casa decimal, no ponto do mês
                      xytext=(0, 8), ha="center", fontsize=8)                         # 8 pontos acima do ponto, centralizado, letra pequena
for posicao, valor in enumerate(tabela_grafico["apostadores_mi"]):                  # o mesmo pro segundo gráfico
    eixos[1].annotate(f"{valor:.1f}", (posicao, valor), textcoords="offset points",   # mesmo texto e posição do gráfico de cima
                      xytext=(0, 8), ha="center", fontsize=8)

eixos[0].set_title("Depósitos nas bets por mês (R$ bilhões)", fontweight="bold")   # título do gráfico de cima
eixos[0].set_ylabel("R$ bilhões", fontweight="bold")                                 # legenda do eixo vertical de cima
eixos[0].set_ylim(0, tabela_grafico["deposito_bi"].max() * 1.15)                     # eixo de 0 até o maior valor + 15%, pro número não cortar
eixos[1].set_title("Número de apostadores por mês (milhões de pessoas)", fontweight="bold")   # título do gráfico de baixo
eixos[1].set_ylabel("Milhões de pessoas", fontweight="bold")                         # legenda do eixo vertical de baixo
eixos[1].set_ylim(0, tabela_grafico["apostadores_mi"].max() * 1.15)                  # mesma folga de 15%
eixos[1].set_xlabel("Mês", fontweight="bold")                                        # legenda do eixo horizontal

plt.suptitle(f"Apostas no Brasil: evolução mensal ({tabela_bets['mes'].iloc[0]} a {tabela_bets['mes'].iloc[-1]})\nFonte: Ministério da Fazenda (dados do endpoint do grupo)",   # título geral em 2 linhas, com o período que veio dos dados
             fontweight="bold", fontsize=12)                                         # negrito, tamanho 12
plt.xticks(range(len(tabela_grafico)), tabela_grafico.index, rotation=45, ha="right")   # mostra TODOS os meses (sem isso o gráfico pula alguns), inclinados em 45 graus
plt.tight_layout()                                                                   # ajusta as margens pra nada ficar cortado
plt.savefig("/content/grafico_bets_evolucao.png", bbox_inches="tight")               # salva a imagem
plt.show()                                                                           # mostra o gráfico na tela

---
# Dados do IBGE: população e renda por estado

**O que este notebook faz**
1. Baixa do endpoint (mockapi) os dados do IBGE por estado: população, renda per capita, IDH, área e outros.
2. Deixa a pessoa escolher o estado num menu e clicar em "Analisar" pra ver a evolução da população e da renda per capita.

**De onde vêm os dados**

Do endpoint (mockapi) montado com dados do IBGE.

**Limites conhecidos**
- A renda per capita de 2026 não existe ainda (o IBGE não divulgou), por isso o gráfico vai de 2022 a 2025.
- A população de 2023 é igual à de 2022 (o IBGE não publicou estimativa de 2023). Por isso o gráfico marca "2023*".
- O IBGE calculou a renda de 2022 com a 5ª visita da Pesquisa PNAD e a de 2023 em diante com a 1ª visita. Então comparar 2022 com 2023 não é perfeito.

---
Bibliotecas do menu e do botão

In [ ]:
import matplotlib.ticker as ticker                     # formata os números do eixo (ex.: R$ 1.000)
import ipywidgets as widgets                           # cria o menu (dropdown) e o botão
from IPython.display import display, clear_output      # display mostra o menu/botão; clear_output apaga a resposta anterior

---
Configurações do IBGE

In [ ]:
# Endereço do endpoint (mockapi) com os dados do IBGE. Devolve um JSON com um item por estado.
url_mockapi = "https://6aa1d1d4ccb3db9689a5cff3.mockapi.io/densidade-ibge-2026"

---
Download dos dados

In [ ]:
print("Fazendo requisição HTTP para o endpoint JSON...")       # avisa que vai começar

res = requests.get(url_mockapi, timeout=30)                    # faz o pedido (GET) e guarda a resposta em res; timeout=30 = desiste se não responder em 30 segundos

if res.status_code == 200:                                     # 200 = deu tudo certo
    dados_json = res.json()                                    # pega o conteúdo da resposta (lista de estados)
    df_geral = pd.DataFrame(dados_json)                    # transforma a lista numa tabela: cada estado vira uma linha

    print(f"Total de estados encontrados: {len(df_geral)}")                      # deveria ser 27 (26 estados + Distrito Federal)
    print(f"Estados disponíveis: {', '.join(df_geral['UF'].unique())}\n")        # junta os nomes numa frase só, separados por vírgula

else:                                                          # se não for 200, algo deu errado
    print(
        f"Erro ao acessar a API. Código: {res.status_code}"    # mostra o código do erro (404, 500...)
    )

---
Gráfico: população e renda por estado (menu e botão)

In [ ]:
# Menu pra escolher o estado, em ordem alfabética.
seletor_estado = widgets.Dropdown(
    options = sorted(df_geral["UF"].unique()),         # lista de estados sem repetir, em ordem alfabética
    description = "Estado:",                           # texto ao lado do menu
    style = {"description_width": "initial"}           # deixa o texto ocupar só o espaço que precisa
)

# Botão que dispara a análise.
pesquisar = widgets.Button(
    description="Analisar",                            # texto do botão
    button_style = "success",                          # cor verde
    icon="bar-chart"                                   # desenho de gráfico dentro do botão
)

saida = widgets.Output()                               # área onde aparecem os textos e o gráfico do estado escolhido

def gerar_analise(b):                                  # função que roda toda vez que o botão é clicado
    with saida:                                        # tudo que for impresso/desenhado aqui vai pra área "saida"
        clear_output(wait=True)                        # apaga o resultado anterior (wait=True espera o novo ficar pronto, pra não piscar)
        uf_pesquisa = seletor_estado.value             # estado que está selecionado no menu
        df_uf = df_geral[df_geral['UF'].str.casefold() == uf_pesquisa.casefold()]   # filtra a tabela: só a linha desse estado (casefold ignora maiúscula/minúscula)

        if not df_uf.empty:                            # se achou o estado...
            info = df_uf.iloc[0]                       # pega essa linha (iloc[0] = a primeira e única)
            print(f"\nEstado: {info['UF']}")                                                       # nome do estado
            print(f"Área territorial: {info.get('Area-territorial')} km²")                         # área em km²
            print(f"IDH (2021): {info.get('IDH-2021')}")                                           # índice de desenvolvimento humano
            print(f"Densidade demográfica (2022): {info.get('Densidade-demografica-2022')} hab/km²")   # habitantes por km²

            anos = ['2022', '2023', '2024', '2025']        # anos do gráfico (a renda de 2026 ainda não existe)

            # Usa info[...] no lugar de info.get(..., 0): se o dado de um ano faltar, NÃO vira zero.
            # Fica vazio (NaN) e o gráfico mostra um buraco na linha, em vez de uma queda falsa até 0
            populacao_milhoes = [info[f'Populacao-estado-{ano}'] / 1e6 for ano in anos]   # população de cada ano, em milhões
            renda_per_capita = [info[f'Renda-per-capta-{ano}'] for ano in anos]           # renda de cada ano, em reais

            rotulo_anos = [ano if ano != '2023' else '2023*' for ano in anos]             # troca "2023" por "2023*" (o asterisco é explicado na nota)

            df_serie = pd.DataFrame({
                'Ano': rotulo_anos,                        # anos (2023 com asterisco)
                'Populacao_Milhoes': populacao_milhoes,    # coluna da população
                'Renda_Per_Capta': renda_per_capita        # coluna da renda
            })

            print("\nEVOLUÇÃO DEMOGRÁFICA E ECONÔMICA:")                                         # título da tabela
            print(df_serie.round(2))                                                            # mostra a tabela com 2 casas decimais
            print("* 2023: no endpoint a população é igual à de 2022 (não há valor próprio pra 2023).")   # nota do asterisco

            fig, ax1 = plt.subplots(figsize=(10, 5))       # cria a área do gráfico (eixo da esquerda = ax1)

            # .plot() do pandas desenha as duas linhas de uma vez.
            # set_index('Ano') faz o ano ir pro eixo horizontal | secondary_y = a população usa o eixo da direita (escalas muito diferentes)
            # color = uma cor pra cada linha | style = linha cheia pra renda e tracejada pra população | legend=False = a legenda é montada mais abaixo
            df_serie.set_index('Ano')[['Renda_Per_Capta', 'Populacao_Milhoes']].plot(
                ax=ax1, secondary_y='Populacao_Milhoes', color=['#1f77b4', '#e67e22'],
                style=['-', '--'], linewidth=3, legend=False)
            ax2 = ax1.right_ax                             # o eixo da direita que o pandas criou

            color = '#1f77b4'                                                                                        # azul (renda)
            ax1.set_xlabel('Ano de Referência', fontweight='bold', fontsize=11)                                      # legenda do eixo horizontal
            ax1.set_ylabel('Renda Média Per Capita (R$)', color=color, fontweight='bold', fontsize=11)              # legenda do eixo da esquerda
            ax1.tick_params(axis='y', labelcolor=color)                                                              # números do eixo na mesma cor da linha
            ax1.yaxis.set_major_formatter(ticker.StrMethodFormatter('R$ {x:,.0f}'))                                  # escreve "R$ 1,000" em vez de 1000

            #Linha 2: População em Milhões (Eixo Direito - Demografia)
            color = '#e67e22'                                                                                        # laranja (população)
            ax2.set_ylabel('População Total (Milhões de Hab.)', color=color, fontweight='bold', fontsize=11)         # legenda do eixo da direita
            ax2.tick_params(axis='y', labelcolor=color)                                                              # números do eixo na mesma cor da linha

            line1 = ax1.get_lines()                        # linha da renda (desenhada no eixo da esquerda)
            line2 = ax2.get_lines()                        # linha da população (desenhada no eixo da direita)
            line1[0].set_label('Renda Per Capita (R$)')    # nome da linha azul na legenda
            line2[0].set_label('População (Milhões)')      # nome da linha laranja na legenda

            plt.title(f'Potencial Financeiro e Demográfico - {info["UF"]}\n(Subsídio para Análise de Crédito Regional)', fontsize=13, fontweight='bold', pad=15)   # título em 2 linhas, com o nome do estado

            #Estilo do gráfico
            ax1.grid(axis='y', linestyle='-', linewidth=0.7, color='#e0e0e0')    # grade só na horizontal
            ax1.spines['top'].set_visible(False)                                 # tira a borda de cima do gráfico
            ax2.spines['top'].set_visible(False)                                 # idem no eixo da direita

            #Legendas combinadas embaixo
            lines = line1 + line2                                                # junta as duas listas de linhas
            labels = [l.get_label() for l in lines]                              # pega o nome de cada uma
            ax1.legend(lines, labels, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=10)   # legenda centralizada, abaixo do gráfico, sem moldura

            plt.tight_layout()                             # ajusta as margens pra nada ficar cortado
            plt.show()                                     # mostra o gráfico
        else:                                              # se não achou o estado...
            print(f"Estado '{uf_pesquisa}' não encontrado")

pesquisar.on_click(gerar_analise)                                   # liga o botão à função: ao clicar, roda gerar_analise
display(widgets.HBox([seletor_estado, pesquisar]), saida)           # HBox põe o menu e o botão lado a lado; a saída (gráfico) fica embaixo